# Feedforward Network



## PyTorch Implementation

```
x -> linear -> relu -> linear -> output
```

In [24]:
import torch 
import torch.nn as nn
import torch.optim as optim


class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, X):
        return self.net(X).squeeze(1)

In [37]:
torch.manual_seed(0)
N, D = 100, 4
max_epochs = 200

X = torch.randn(N, D)
y = (torch.sum(X, dim=1) >= 0.5).float()


#initialize model, optimizer and loss function
model = MLP(D, 10, 1)
optimizer = optim.SGD(model.parameters(), lr=1e-1)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(max_epochs):
    optimizer.zero_grad()
    
    logits = model(X)
    loss = loss_fn(logits, y)
    
    loss.backward()
    optimizer.step()

    pred = (torch.sigmoid(logits) >= 0.5).float()
    acc = (pred == y).float().mean()
    if epoch% 100 == 0:
        print(f"loss={loss.item()}, acc={acc.item():.4f}")

loss=0.6929007172584534, acc=0.5200
loss=0.3386307656764984, acc=0.9200


In [12]:
y = torch.randn(N) >= 0.5
y.shape

torch.Size([100])

### NumPy Implementation

```
x -> W1 -> ReLU -> W2 -> Softmax
```
- $X \in \mathbb{R}^{N\times D}$, $D$ is the input dimension
- $W_1 \in \mathbb{R} ^{D\times H}$, $H$ is the hidden layer size
- $W_2 \in \mathbb{R} ^{H\times C}$, $C$ is number of classes
  
first layer:
$$Z_1 = XW_1 + b1$$
$$A_1 = ReLU(Z_1)$$
second layer:
$$Z_2 = A_1W_2 + b2$$
softmax:
$$p = softmax(Z_2)$$
cross-entropy loss
$$L = \frac{1}{N}\sum^N_{i=1} - log(p_{y_i})$$
- $p_{y_i}$ = predicted probability of correct class


#### Gradient of Softmax + Cross Entropy
Derivative simplifies to:
$$\frac{\partial L}{\partial z_2} = p-y$$

- $p$ = predicted probability vector
- $y$ = one-hot true label
This compute
$$dz_2 = p - y$$

This computes
```
dz2 = probs
dz1[range(len(X)), y] -= 1
```


#### Gradient for W2
$$\frac{\partial L}{\partial W_2} = A_1^Tdz_2$$


#### Gradient for ReLU

$$dz_1 = (dz_2 W_2^T)\odot ReLU'(Z_1)$$

```
dz1 = dz2 @ W2.T * (a1 > 0)
```

#### Gradient for W1
$$\frac{\partial L}{\partial W_1} = X^T dz1$$

```
dW1 = X.T @ dz1
```

In [18]:
import numpy as np

def relu(x):
    return np.maximum(0, x)

def softmax(x):
    x = x - np.max(x, axis=1, keepdims=True) #numerically stable softmax

    exp_z = np.exp(x)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

class FeedForward:
    def __init__(self, input_size, hidden_size, output_size, max_epoch=20, lr=0.1):
        self.params = {}
        self.params['W1'] = np.random.rand(input_size, hidden_size)*0.01
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = np.random.rand(hidden_size, output_size)*0.01
        self.params['b2'] = np.zeros(output_size) 
        self.max_epoch = max_epoch
        self.lr = lr

    def forward(self, X):
        W1, b1 = self.params['W1'], self.params['b1']
        W2, b2 = self.params['W2'], self.params['b2']

        z1 = X @ W1 + b1 #NxH
        a1 = relu(z1)
        z2 = a1 @ W2 + b2 #NxC
        probs = softmax(z2)
        return probs

    
    def train(self, X, y):
        N, D = X.shape
        C = np.max(y) + 1
        
        for epoch in range(self.max_epoch):
            # pred = self.forward(X)
            W1, b1 = self.params['W1'], self.params['b1']
            W2, b2 = self.params['W2'], self.params['b2']
    
            z1 = X @ W1 + b1 #NxH
            a1 = relu(z1)
            z2 = a1 @ W2 + b2 #NxC
            probs = softmax(z2)
            
            # backprop    
            dz2 = probs.copy()
            dz2[np.arange(N), y] -= 1
            dz2 /= N

            dW2 = a1.T @ dz2
            db2 = np.sum(dz2, axis=0)
            
            dz1 = (dz2 @ W2.T) * (z1 > 0)
            dW1 = X.T @ dz1
            db1 = np.sum(dz1, axis=0)

            self.params['W1'] -= self.lr * dW1 
            self.params['W2'] -= self.lr * dW2 
            self.params['b1'] -= self.lr * db1
            self.params['b2'] -= self.lr * db2 
        
        
    

In [20]:
# Generate a toy dataset
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([0, 1, 1, 0])

# Initialize a neural network
net = FeedForward(input_size=2, hidden_size=10, output_size=2, max_epoch=100)

# Train the neural network
net.train(X, y)

# Test the neural network
probs = net.forward(X)
predictions = np.argmax(probs, axis=1)
print("Predictions: ", predictions)

Predictions:  [0 1 1 0]


## Improvements

### Weight initialization
Xavier initialization to improve convergence and avoid vanishing or exploding gradients. One possible implementation for Xavier initialization of the weights is:

In [25]:

# Xavier initialization
input_size, hidden_size, output_size = 10, 2, 2
W1 = np.random.randn(input_size, hidden_size) / np.sqrt(input_size)
W2 = np.random.randn(hidden_size, output_size) / np.sqrt(hidden_size)

### Learning rate decay
A common technique is to gradually decrease the learning rate over time, known as learning rate decay, to fine-tune the network weights as the optimization process progresses.

In [29]:
# Learning rate decay
num_epochs = 10
learning_rate = 0.1
lr_decay = 0.95
lr_decay_epoch = 100
for epoch in range(num_epochs):
    # ...
    if epoch % lr_decay_epoch == 0:
        learning_rate *= lr_decay

### Regularization
Regularization techniques such as L1 or L2 regularization can be applied to the loss function to prevent overfitting and improve the generalization performance of the model.  

#### L2 Regularization

$$L = \frac{1}{N}\sum - logp_y + \frac{\lambda}{2}(\Vert W_1 \Vert^2 + \Vert W_2 \Vert^2)$$

$$\Vert W \Vert^2 = \sum W_{ij}^2$$

In [ ]:
data_loss = 0
reg = 0.1
dW2 = a1.T @ dz2 + reg * W2
dW1 = X.T @ dz1 + reg * W1

#Loss
loss = data_loss + 0.5 * reg * (np.sum(W1 ** 2) + np.sum(W2 ** 2))

#### L1 Regularization

$$L = L_{data} + \lambda \sum |W| = L_{data} + \lambda(\sum |W_1| + \sum |W_2|)$$ 
$$\nabla = sign(W)$$
During backprop:

$$\frac{\partial L}{\partial W} = \frac{\partial L_{data}}{\partial W} + \lambda \cdot sign(W)$$
$$W = W - \eta (\nabla_{data} + \lambda sign(W))$$

Even if the data gradient is tiny, the regularization still pushes it to 0 -> sparser model  
However, L2 regularization rarely reaches it

In [ ]:
dW2 = a1.T @ dz2 + reg * np.sign(W2)
dW1 = X.T @ dz1 + reg * np.sign(W1)

loss = data_loss + reg * (np.sum(np.abs(W1)) + np.sum(np.abs(W2))


### Mini-batch training

In [ ]:
# Mini-batch training
batch_size = 64
num_batches = len(X) // batch_size
for epoch in range(num_epochs):
    for i in range(num_batches):
        # Select a random batch of data
        batch_mask = np.random.choice(len(X), batch_size)
        X_batch = X[batch_mask]
        y_batch = y[batch_mask]

        # Forward and backward propagation using the batch data
        # ...

### Optimization

1. **SGD**:
    $$\theta = \theta - \eta \nabla_\theta L$$
2. **SGD with Momentum**: adds a velocity term to smooth updates. accelerate in consistent direction and reduce oscillation
    $$v_t = \beta v_{t-1} + \nabla L$$
    $$\theta = \theta - \eta v_t$$


In [35]:
class MomentumSGD:
    def __init__(self, lr, beta):
        self.lr = lr
        self.beta = beta
        self.v = {}
    def step(self, params, grads):
        for k in params:
            if k not in self.v:
                self.v[k] = np.zeros_like(params[k])
            self.v[k] = self.beta * self.v[k] + grads[k]
            self.params[k] -= self.lr * self.v[k]

3. **Adam optimizer**:
    1. Momentum (first moment)
    2. RMSProp (second moment)
    3. Bias correction

First moment (mean):
$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
Second moment (variance):
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$

Bias correction:
$$\hat{m_t} = \frac{m_t}{1 - \beta^t_1}$$
$$\hat{v}_t = \frac{v_t}{1 - \beta^t_2}$$

Update:
$$\theta = \theta - \eta \frac{\hat{m_t}}{\sqrt{\hat{v}_t} + \epsilon}$$

In [ ]:
# Adam optimization
beta1, beta2 = 0.9, 0.999
eps = 1e-8
mW1, vW1 = 0, 0
mW2, vW2 = 0, 0
for epoch in range(num_epochs):
    # Forward and backward propagation
    # ...
    # Update parameters using Adam optimization
    mW1 = beta1 * mW1 + (1 - beta1) * dW1
    vW1 = beta2 * vW1 + (1 - beta2) * (dW1 ** 2)
    mW2 = beta1 * mW2 + (1 - beta1) * dW2
    vW2 = beta2 * vW2 + (1 - beta2) * (dW2 ** 2)
    self.params['W1'] -= learning_rate * mW1 / (np.sqrt(vW1) + eps)
    self.params['b1'] -= learning_rate * db1
    self.params['W2'] -= learning_rate * mW2 / (np.sqrt(vW2) + eps)
    self.params['b2'] -= learning_rate * db2